# 07 · Serving

**Story so far:** chapter 06 produced a versioned sentiment model artifact. Tasks run to
completion; **apps** stay up and serve traffic. Same environment model (image, resources,
secrets), plus a URL, autoscaling, and rollout management. This chapter puts Review
Radar's model behind a live API — wired to the training task's output *by name*, so every
redeploy picks up the latest trained model automatically.

**Learning goals**

1. Deploy the model-serving app and call it; understand the deploy → activate lifecycle
2. See the train → serve wiring: `Parameter` + `RunOutput` resolves the model artifact at
   deploy time
3. Configure autoscaling (including scale-to-zero) and measure cold vs warm latency
4. (Optional) serve an LLM with the vLLM app environment

> **Why the app code is in `scripts/`:** app deployment needs a deployable file — it isn't
> supported from interactive notebook sessions. The notebook *drives* the deploys and then
> exercises the endpoints. Open the scripts side-by-side as you go:
> [`fastapi_app.py`](./scripts/apps/fastapi_app.py) ·
> [`streamlit_app.py`](./scripts/apps/streamlit_app.py) ·
> [`vllm_app.py`](./scripts/apps/vllm_app.py)

In [ ]:
import flyte

flyte.init_from_config()

## 1. The train → serve wiring

Open [`scripts/apps/fastapi_app.py`](./scripts/apps/fastapi_app.py). Three things to
notice:

1. **The model arrives as a parameter, not a copy.** The app declares:

   ```python
   Parameter(
       name="model",
       value=RunOutput(type="file", task_name="training.train_model",
                       task_auto_version="latest"),
       download=True,
       env_var="MODEL_PATH",
   )
   ```

   At deploy time, Flyte resolves the **latest run of chapter 06's training task**,
   downloads its output artifact, and hands the app its local path via `MODEL_PATH`.
   Retrain → redeploy → new model. No manual copying, no drift between "the model we
   trained" and "the model we serve".

2. **The model loads once, at startup** (FastAPI lifespan) — never per request.

3. **There's a fallback.** If no training run exists yet, the app serves a keyword
   heuristic and reports `model_kind: heuristic-fallback`, so this chapter also works
   standalone.

Make sure chapter 06's `train_model` has run at least once, then deploy:

In [ ]:
!python scripts/apps/fastapi_app.py

In [ ]:
# Paste the URL printed above:
APP_URL = "https://REPLACE-ME"

import httpx

print(httpx.get(f"{APP_URL}/health", timeout=120).json())
r = httpx.post(f"{APP_URL}/score",
               json={"text": "absolutely love this espresso machine"}, timeout=120)
r.json()

`model_kind: trained` confirms the app is serving the chapter-06 artifact.

## 2. Cold vs warm latency

The app's `Scaling(replicas=(0, 2), scaledown_after=300)` scales to zero when idle, so
the first request after idle pays the **cold start** (schedule + image pull + process
boot + model load). Subsequent requests hit a warm replica:

In [ ]:
import time

def probe(n: int = 10) -> list:
    times = []
    for i in range(n):
        t0 = time.perf_counter()
        httpx.post(f"{APP_URL}/score", json={"text": f"probe review {i}"}, timeout=120)
        times.append((time.perf_counter() - t0) * 1000)
    return [round(t, 1) for t in times]

latencies = probe()
print(f"first (cold-ish): {latencies[0]} ms")
print(f"warm p50: {sorted(latencies[1:])[len(latencies)//2]} ms")
latencies

**The production trade-off in one line:** `replicas=(0, N)` minimizes cost,
`replicas=(1, N)` buys away the cold start. For latency-sensitive inference keep min ≥ 1
and use `scaledown_after` to control how aggressively extra replicas retire.

Other levers on `AppEnvironment` worth knowing:

| Lever | Effect |
|---|---|
| `scaling=flyte.app.Scaling(replicas=(min, max), scaledown_after=...)` | Replica autoscaling |
| `requires_auth=True` (default) | Endpoint requires platform auth; `False` opens it up |
| `parameters=[Parameter(...)]` | The train→serve wiring above |
| `env_vars`, `secrets` | Same as tasks |
| custom domains | Serve under your own hostname (platform config) |

Manage lifecycle from the CLI:

```bash
flyte get app                       # list apps + status
```

Apps roll out per-version; deactivating stops traffic without deleting history.

> 💬 **Discuss:** for each service the customer plans to run, what does a cold start cost
> (user-facing latency) vs an always-warm minimum replica (steady spend)? Where does each
> land on `replicas=(0, N)` vs `(1, N)`?

## 3. Optional: LLM serving with vLLM

`VLLMAppEnvironment` wraps the vLLM server: point it at a HuggingFace model, give it a
GPU, and you get an **OpenAI-compatible** endpoint. The workshop default is a 0.5B model
that fits any small GPU — chapter 08's agent can use it as a fully self-hosted LLM.

In [ ]:
# Requires a schedulable GPU. HF_MODEL_ID comes from .env (default Qwen2.5-0.5B).
!python scripts/apps/vllm_app.py

In [ ]:
LLM_URL = "https://REPLACE-ME"      # from the cell above

r = httpx.post(
    f"{LLM_URL}/v1/chat/completions",
    json={
        "model": "review-radar-llm",
        "messages": [{"role": "user", "content":
                      "One sentence: summarize the sentiment of 'terrible shoes, arrived broken'."}],
        "max_tokens": 60,
    },
    timeout=300,
)
r.json()["choices"][0]["message"]["content"]

Inference tuning levers (on the app environment / vLLM args): throughput via continuous
batching (`--max-num-seqs`, `--tensor-parallel-size N` with multiple GPUs), cold start via
`disk` sizing + model prefetching (docs → *serve-and-deploy-apps → prefetching models*),
cost via scale-to-zero.

> 💬 **Discuss:** which GPU types does this deployment actually offer, and which models on
> the customer's roadmap fit them? (This decides `gpu=` for every serving app.)

## 4. Dashboards, and apps calling tasks

Any process that binds a port can be an app (`args=[...]` + `port=...`) — the
[Streamlit dashboard](./scripts/apps/streamlit_app.py) ships the same way:

In [ ]:
!python scripts/apps/streamlit_app.py

And the loop closes in both directions:

- **Task → app:** just HTTP (`httpx` from any task).
- **App → task:** an app can launch runs — e.g. a "retrain now" button:

```python
@app.post("/retrain")
async def retrain():
    run = flyte.with_servecontext().run(train_model, n_examples=2000, c=1.0)
    return {"run_url": run.url}
```

**Story checkpoint:** Review Radar's model is live, self-updating from training runs, and
measurable. One chapter of new functionality left: put an *agent* in front of it.

## Further reading

- Next: [08-agents-and-sandboxing](./08-agents-and-sandboxing.ipynb)
- Union docs → User guide → *Build apps*, *Configure apps*, *Serve and deploy apps*